# One frame stack → spots → angles

Start here for a **single Polarcam frame stack and its matching minimum background**. Change the two paths below, then **Run All**. No camera or dated analysis folders are required by the code. The example paths point to the first 12 October hinge stack.

1. Check the background and saturation provenance.
2. Detect spots on one exposure; inspect retained and rejected candidates.
3. Compare the pixels, X/Y and intensity before/after subtraction.
4. Plot one r and θ per retained spot from the selected exposure.

**Starting choices:** frame 1, 2 SD intensity detection, 18×18 extraction, no upper component size limit, strongest-first overlap rejection. No temporal averaging, fitted gains, survival filter or hidden region mask. Detection is the GUI's **intensity DoG method with these analysis settings**, not its motion detector or its default 8 SD / 250-pixel limit. Results are candidate detections, not confirmed individual rods.

Setup once, in PowerShell in this folder:
```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[analysis]"
.\.venv\Scripts\python.exe -m jupyterlab Framestack_quickstart.ipynb
```
Use this checkout's Python environment as the notebook kernel. Input files are read-only. Export is off by default.

In [ ]:
from pathlib import Path
import sys, json, hashlib, csv
from collections import Counter
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse
from matplotlib.ticker import PercentFormatter
from IPython.display import display, HTML

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/polarcam/optics.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the polarcam folder.")
sys.path.insert(0, str(ROOT / "src"))
from polarcam.live.Detection_alg_offline import intensity_detection_map, _finite_stats
from polarcam.live.intensity_spots import channel_means, anisotropy
from polarcam.live.recording_io import read_recording_metadata, recording_fps, recording_phase_marker, strip_phase_marker
from polarcam.optics import theta_model, theta_from_radius, pair_sum_imbalance

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

## 1. Inputs and settings

Only the two `.npy` files are required. Keep the stack's JSON and `diagnostics/` companion beside it when available: these allow automatic BG/phase detection and an exact pre-subtraction saturation check.

With **no JSON**, set `SAVED_WITH_BG` explicitly: `True` for already-subtracted pixels, `False` for raw pixels. For a cropped image also supply its sensor origin if unknown. An appended Polarcam phase marker is detected and removed; it is never a measurement.

In [ ]:
DAY = ROOT / "data/hinges/polarcam/2026-10-12"
FRAMESTACK = DAY / "frames/frame_stack_20261012-183850.npy"
MINIMUM_BG = DAY / "backgrounds/background_min_20261012-183650_exp1.2ms_gain1_bg_widefield.npy"

FRAME_NUMBER = 1                 # 1-based: detection AND population angles use this exposure
K_SD = 2.0                      # Try 4 or 8, then rerun from detection onwards
MIN_AREA = 10                    # Sensor pixels in the thresholded connected component
MAX_AREA = None                  # None permits oversized components
APERTURE = 18                    # Even square, sensor pixels
DISPLAY_GAIN = 20                # Display only; never applied to intensities or angles
EXCLUDE_ELLIPSES = []            # Optional local-image (cx, cy, rx, ry) tuples
# The old 12 October corner mask was [(2060, 500, 320, 220)]. It is NOT automatic here.

SAVED_WITH_BG = None             # None: read JSON. Without JSON, set True or False
ORIGIN_XY = None                 # None: metadata/marker; otherwise sensor (x, y)
PHASE_MARKER = None              # None: metadata/legacy detection; override True/False if needed
FPS = None                      # None: metadata; without it traces use frame number
RAW_FULL_SCALE = 4095            # Native Mono12; do not use an 8-bit display-export stack
SATURATION_AT = 4094             # Conservative threshold used in our analysis

MODEL = theta_model(n_sample=1.33, na_out=1.30, na_in=0.39, n_immersion=1.515)
A, B, C, R_LIMIT = [MODEL[k] for k in ("A", "B", "C", "r_max")]

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024**2), b""): h.update(block)
    return h.hexdigest()

def valid_image(a, name):
    if a.ndim != 2 or not np.isfinite(a).all() or (a < 0).any():
        raise ValueError(f"{name} must be a finite nonnegative native-pixel image.")

metadata = read_recording_metadata(FRAMESTACK)
meta = metadata[0] if metadata else {}
bg_meta = meta.get("background", {})
output_meta = meta.get("fetch_frames_output", {})
recorded_bg = bg_meta.get("background_subtracted", output_meta.get("background_subtracted"))
subtracted = recorded_bg if SAVED_WITH_BG is None else SAVED_WITH_BG
if not isinstance(subtracted, bool):
    raise ValueError("No saved-BG flag: set SAVED_WITH_BG to True or False above, then rerun.")
if isinstance(recorded_bg, bool) and SAVED_WITH_BG is not None and recorded_bg != SAVED_WITH_BG:
    raise ValueError("SAVED_WITH_BG conflicts with the recorded JSON; check the input, do not subtract twice.")

stored = np.load(FRAMESTACK, mmap_mode="r", allow_pickle=False)
if stored.ndim != 3 or stored.dtype == np.uint8:
    raise ValueError("Use a native grayscale stack with shape (frames, height, width), not an 8-bit preview.")
phase = {}
marker = recording_phase_marker(metadata) if PHASE_MARKER is None else PHASE_MARKER
movie = strip_phase_marker(stored, marker_appended=marker, roi_meta=phase)
n_frames, height, width = movie.shape
if n_frames < 1 or min(height, width) < 32 or height % 2 or width % 2:
    raise ValueError("Need nonempty frames with even dimensions, at least 32×32.")
expected_n = meta.get("actual", {}).get("frames")
if expected_n is not None and int(expected_n) != n_frames:
    raise ValueError("Measurement count disagrees with the JSON; check the phase-marker setting.")
if not 1 <= FRAME_NUMBER <= n_frames: raise ValueError("FRAME_NUMBER is outside this stack.")
roi = meta.get("roi") or output_meta.get("roi") or {}
origin = ORIGIN_XY
if origin is None:
    for xkey, ykey in [("OffsetX", "OffsetY"), ("x", "y")]:
        if xkey in roi and ykey in roi:
            origin = (int(roi[xkey]), int(roi[ykey])); break
absolute_origin_known = origin is not None
if origin is None:
    origin = (phase.get("phase_x", 0), phase.get("phase_y", 0))
if len(origin) != 2 or any(int(v) != v or v < 0 for v in origin):
    raise ValueError("ORIGIN_XY must contain nonnegative integer sensor offsets.")
origin = tuple(map(int, origin))
if phase and (origin[0] % 2, origin[1] % 2) != (phase['phase_x'], phase['phase_y']):
    raise ValueError("Sensor origin conflicts with the polarization phase marker.")

bg_file_hash = sha256(MINIMUM_BG)
expected_hash = bg_meta.get("background_profile_sha256") or output_meta.get("background_profile_sha256")
if subtracted and expected_hash and bg_file_hash.lower() != expected_hash.lower():
    raise ValueError("This is not the background used for the saved subtraction (SHA-256 mismatch).")
bg = np.load(MINIMUM_BG, allow_pickle=False)
valid_image(bg, "Background")
if bg.shape != (height, width):
    if not absolute_origin_known:
        raise ValueError("A full-sensor BG with a cropped movie needs the absolute ORIGIN_XY.")
    ox, oy = origin; bg = bg[oy:oy + height, ox:ox + width]
if bg.shape != (height, width): raise ValueError("Background does not cover the stack ROI.")
bg = np.asarray(bg, dtype=np.float64)
fps = recording_fps(metadata) if FPS is None else float(FPS)
if fps is not None and (not np.isfinite(fps) or fps <= 0): raise ValueError("FPS must be positive.")
time = np.arange(n_frames) / fps if fps else np.arange(1, n_frames + 1)
time_label = "Time (s, inferred from FPS)" if fps else "Frame number"

def images(frame_index):
    saved = np.asarray(movie[frame_index], dtype=np.float64)
    valid_image(saved, "Frame")
    corrected = saved if subtracted else np.maximum(saved - bg, 0)
    raw = saved + bg if subtracted else saved
    return raw, corrected

raw_label = "Raw proxy: BG added back" if subtracted else "Raw camera pixels"
print(f"{n_frames} measurement frames; removed marker: {len(stored) != n_frames}; saved BG subtraction: {subtracted}")
print(f"Polarization origin/phase: {origin}; BG: {MINIMUM_BG.name}; SHA-256: {bg_file_hash[:12]}")
print("BG identity:", "matches saved metadata" if subtracted and expected_hash else "user-supplied; not verified against a saved subtraction hash")
if not absolute_origin_known: print("Using marker parity, or full-frame origin (0,0) if no marker. Check ORIGIN_XY for legacy crops.")
print(f"Model A={A:.8f}, B={B:.8f}, C={C:.8f}; r limit={R_LIMIT:.6f}")

### Saturation before background subtraction

Use the hash-verified **raw maximum companion** when present. Otherwise scan every measurement frame; for already-subtracted stacks, add back the supplied minimum BG. Zero-clipped values cannot be recovered exactly, so the reconstructed maximum is an **upper bound** there. Without the companion, flags are conservative potential saturation exclusions, conditional on supplying the actual subtraction BG. This is not a new raw movie.

A spot is excluded if **any pixel in its extraction aperture or detected component bounding box** reaches `SATURATION_AT` in **any** measurement frame. Display clipping at 20× is different from sensor saturation.

In [ ]:
if not 0 < SATURATION_AT <= RAW_FULL_SCALE: raise ValueError("Check the saturation scale.")
diagnostic = output_meta.get("raw_peak") or meta.get("actual", {}).get("raw_peak")
diagnostic_path = FRAMESTACK.parent / diagnostic["path"] if diagnostic else None
if diagnostic_path is not None and diagnostic_path.is_file():
    if sha256(diagnostic_path) != diagnostic["sha256"]:
        raise ValueError("Raw-peak companion hash mismatch.")
    sensor_peak = np.load(diagnostic_path, allow_pickle=False)
    if sensor_peak.shape != (height, width) or int(diagnostic["frames"]) != n_frames:
        raise ValueError("Raw-peak companion has the wrong shape or frame count.")
    if float(diagnostic["full_scale"]) != RAW_FULL_SCALE:
        raise ValueError("RAW_FULL_SCALE disagrees with the raw diagnostic.")
    saturation_source = "Exact raw-peak companion, hash verified"
else:
    saved_peak = np.zeros((height, width), dtype=np.float64)
    for frame in movie:
        valid_image(frame, "Frame")
        np.maximum(saved_peak, frame, out=saved_peak)
    rounding_bound = 0.5 if subtracted and not np.array_equal(bg, np.rint(bg)) else 0.0
    sensor_peak = saved_peak + bg + rounding_bound if subtracted else saved_peak
    saturation_source = "BG-added upper bound; conservative exclusion" if subtracted else "Exact raw stack maximum"
valid_image(sensor_peak, "Raw maximum")
print(saturation_source)
print(f"Raw/upper-bound image maximum: {sensor_peak.max():.1f}; flag threshold: {SATURATION_AT}")

raw_frame, corrected_frame = images(FRAME_NUMBER - 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), layout="constrained")
for ax, arr, title in zip(axes, [raw_frame, bg, corrected_frame], [raw_label, "Minimum background", "After subtraction"]):
    ax.imshow(intensity_detection_map(arr), cmap="gray", vmin=0, vmax=RAW_FULL_SCALE / DISPLAY_GAIN)
    ax.set_title(title); ax.set_axis_off()
fig.suptitle(f"Frame {FRAME_NUMBER} · same {DISPLAY_GAIN}× display · 2×2 means for overview only")
plt.show()

## 2. Detection and explicit exclusions

The aligned 2×2 intensity map averages the four analyzer pixels **only for detection/display**. DoG widths are 1 and 6 sensor pixels. Keep connected pixels above the whole filtered image's mean + `K_SD × SD` (and above zero). Native polarization pixels are used for extraction.

The overlap rule keeps the stronger DoG peak when two extraction windows overlap. It does not separate touching rods. The table and red markers retain saturation/overlap/region/size exclusions for inspection. Changing the frame or detector settings requires rerunning this cell and the cells below it.

In [ ]:
if APERTURE < 2 or APERTURE % 2 or APERTURE >= min(height, width) - 2:
    raise ValueError("APERTURE must be an even, positive window smaller than the image.")
if not np.isfinite(K_SD) or K_SD < 0 or MIN_AREA < 1:
    raise ValueError("Use nonnegative K_SD and positive MIN_AREA.")
if MAX_AREA is not None and MAX_AREA < MIN_AREA: raise ValueError("MAX_AREA is below MIN_AREA.")
for cx, cy, rx, ry in EXCLUDE_ELLIPSES:
    if rx <= 0 or ry <= 0: raise ValueError("Ellipse radii must be positive.")
half = APERTURE // 2
margin = max(10, half + 1)
if min(height, width) <= 2 * margin: raise ValueError("Image too small for this aperture and edge exclusion.")
raw_frame, corrected_frame = images(FRAME_NUMBER - 1)
work = intensity_detection_map(corrected_frame)[margin:-margin, margin:-margin]
dog = (cv2.GaussianBlur(work, (0, 0), 1, borderType=cv2.BORDER_REPLICATE)
       - cv2.GaussianBlur(work, (0, 0), 6, borderType=cv2.BORDER_REPLICATE))
mu, sd = _finite_stats(dog)
threshold = max(0, mu + K_SD * sd)
_, labels, stats, centers = cv2.connectedComponentsWithStats((dog > threshold).astype('uint8'), connectivity=8)
spots = []
for component in range(1, len(stats)):
    bx, by, w, h, area = map(int, stats[component])
    if area < MIN_AREA: continue
    x, y = map(float, centers[component] + margin)
    l, t = round(x) - half, round(y) - half
    bounds = [l, l + APERTURE, t, t + APERTURE]
    bbox = [bx + margin, bx + margin + w, by + margin, by + margin + h]
    peak = max(float(sensor_peak[t:t + APERTURE, l:l + APERTURE].max()),
               float(sensor_peak[bbox[2]:bbox[3], bbox[0]:bbox[1]].max()))
    inside = any(((x-cx)/rx)**2 + ((y-cy)/ry)**2 <= 1 for cx,cy,rx,ry in EXCLUDE_ELLIPSES)
    xy_one = anisotropy(channel_means(corrected_frame, [bounds], origin=origin)[0])
    saturated = peak >= SATURATION_AT
    reason = ("size" if MAX_AREA is not None and area > MAX_AREA else "region" if inside else
              "saturated" if saturated else "undefined XY" if not np.isfinite(xy_one).all() else "eligible")
    component_mask = labels[by:by+h, bx:bx+w] == component
    spots.append(dict(id=f"C{component}", component=component, x=x, y=y, bounds=bounds, bbox=bbox,
                      area=area, dog_peak=float(dog[by:by+h, bx:bx+w][component_mask].max()),
                      raw_peak=peak, saturated=saturated, reason=reason, suppressed_by=None))

def overlap(a, b):
    al,ar,at,ab=a['bounds']; bl,br,bt,bb=b['bounds']
    return max(al,bl) < min(ar,br) and max(at,bt) < min(ab,bb)

retained = []
for s in sorted(spots, key=lambda s: (-s['dog_peak'], s['component'])):
    if s['reason'] != 'eligible': continue
    stronger = next((q for q in retained if overlap(s,q)), None)
    s['reason'] = 'overlap' if stronger else 'kept'
    if stronger: s['suppressed_by'] = stronger['id']
    else: retained.append(s)
print(f"DoG threshold {threshold:.3f} native intensity units; {len(spots)} candidates ≥{MIN_AREA} pixels")
print("Exclusions / kept:", dict(Counter(s['reason'] for s in spots)))
print("Candidates with any-frame saturation flag (including other exclusions):", sum(s['saturated'] for s in spots))

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(intensity_detection_map(corrected_frame), cmap='gray', vmin=0, vmax=RAW_FULL_SCALE / DISPLAY_GAIN)
for reason, color in [('kept','#2cd3de'),('excluded','#fa8266')]:
    group = [s for s in spots if (s['reason']=='kept') == (reason=='kept')]
    ax.scatter([s['x'] for s in group], [s['y'] for s in group], s=28, facecolors='none', edgecolors=color, lw=.7, label=f'{reason}: {len(group)}')
for cx,cy,rx,ry in EXCLUDE_ELLIPSES:
    ax.add_patch(Ellipse((cx,cy),2*rx,2*ry,fill=False,color='#fa8266'))
ax.set(title=f"Frame {FRAME_NUMBER} · {K_SD:g} SD · {DISPLAY_GAIN}× display", xlabel='Local sensor x', ylabel='Local sensor y')
ax.legend(); plt.show()

## 3. Extract the original pixels and recover angles

Each fixed aperture gives channel means in **[0°, 90°, 45°, 135°]** order, preserving the sensor phase. The shared `polarcam.optics` model includes **Fresnel transmission, BFP mapping and the circular NA 0.39–1.30 hole mask**. No APD matrix or automatically fitted camera gain correction is applied.

\[
X=\frac{I_0-I_{90}}{I_0+I_{90}},\quad Y=\frac{I_{45}-I_{135}}{I_{45}+I_{135}},\quad
r=\sqrt{X^2+Y^2},\quad \theta=\arcsin\sqrt{\frac{Ar}{C-Br}}.
\]

φ = ½ atan2(Y, X), modulo 180°; it is undefined at r = 0. θ is measured from the surface normal, **not automatically the hinge opening angle**. Above-limit radii stay visible in r, but give `NaN` θ; they are not clipped to 90°. Background subtraction does not itself remove coherent interference. These are model-dependent apparent angles for an ideal dipole with corrected channels; residual background, interference and unresolved multiple rods can bias them. [Full model assumptions](docs/angle_reconstruction.md).

In [ ]:
bounds = [s['bounds'] for s in spots]
channels = np.empty((n_frames, len(spots), 4), dtype=np.float64)
raw_channels = np.empty_like(channels)
for t in range(n_frames):
    raw, corrected = images(t)
    channels[t] = channel_means(corrected, bounds, origin=origin)
    raw_channels[t] = channel_means(raw, bounds, origin=origin)
xy = anisotropy(channels)
raw_xy = anisotropy(raw_channels)
r = np.linalg.norm(xy, axis=-1)
theta_deg = np.degrees(theta_from_radius(r, A, B, C))
phi_deg = np.mod(np.degrees(0.5 * np.arctan2(xy[...,1], xy[...,0])), 180)
phi_deg[r == 0] = np.nan
imbalance = pair_sum_imbalance(channels)  # Diagnostic only, not an automatic gain fit
keep = np.array([s['reason']=='kept' for s in spots], dtype=bool)
for i,s in enumerate(spots):
    s['r'] = float(r[FRAME_NUMBER-1,i])
    s['theta_deg'] = float(theta_deg[FRAME_NUMBER-1,i])
    s['phi_deg'] = float(phi_deg[FRAME_NUMBER-1,i])
    s['intensity_8bit_equiv'] = float(channels[FRAME_NUMBER-1,i].mean()*255/RAW_FULL_SCALE)
    s['pair_sum_imbalance'] = float(imbalance[FRAME_NUMBER-1,i])
print(f"Extracted {n_frames} individual exposures for {len(spots)} candidates; no time averaging.")

## 4. Inspect a spot

Choose a candidate and frame below. The three image panels show **native pixels** at the same display gain; the cyan box is the extraction window. The traces use all individual exposures. White at 20× usually means **display clipping**, not sensor saturation. Change the display gain to 1× to see bright structure.

The inspector does not silently change selection. Saturation is checked over the whole stack, even when inspecting a single frame. Without widgets, call `inspect_spot("C123", frame_number=1, gain=20)` directly. Set `pixel_values=True` to print the native aperture arrays.

In [ ]:
def inspect_spot(spot_id, frame_number=1, gain=20, pixel_values=False):
    i = next(i for i,s in enumerate(spots) if s['id']==spot_id)
    s = spots[i]; t = int(frame_number)-1
    if not 0 <= t < n_frames or gain <= 0: raise ValueError("Invalid frame or display gain.")
    raw, corrected = images(t)
    l,rr,top,bottom = s['bounds']
    x0=max(0,round(s['x'])-32); x1=min(width,x0+64)
    y0=max(0,round(s['y'])-32); y1=min(height,y0+64)
    fig, axes = plt.subplots(2,3,figsize=(13,7.6),layout='constrained')
    for ax,arr,title in zip(axes[0], [raw,bg,corrected], [raw_label,'Minimum background','After subtraction']):
        ax.imshow(arr[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=RAW_FULL_SCALE/gain,extent=(x0,x1,y1,y0),interpolation='nearest')
        ax.add_patch(Rectangle((l,top),rr-l,bottom-top,fill=False,edgecolor='#28becd',lw=1))
        ax.set_title(title); ax.set_axis_off()
    a,b,c = axes[1]
    a.scatter(raw_xy[:,i,0],raw_xy[:,i,1],s=5,alpha=.35,color='.55',label='Before / proxy')
    a.scatter(xy[:,i,0],xy[:,i,1],s=5,alpha=.6,color='#147f9b',label='BG subtracted')
    bg_xy=anisotropy(channel_means(bg,[s['bounds']],origin=origin)[0])
    a.scatter(*bg_xy,marker='x',s=50,c='#c66037',label='BG only')
    angle=np.linspace(0,2*np.pi,300)
    a.plot(R_LIMIT*np.cos(angle),R_LIMIT*np.sin(angle),':',c='.65',lw=.8)
    a.set(xlim=(-1.05,1.05),ylim=(-1.05,1.05),aspect='equal',xlabel='X',ylabel='Y',title='All exposures')
    a.legend(fontsize=8)
    b.plot(time,raw_channels[:,i].mean(axis=-1)*255/RAW_FULL_SCALE,c='.55',lw=.7,label='Before / proxy')
    b.plot(time,channels[:,i].mean(axis=-1)*255/RAW_FULL_SCALE,c='#147f9b',lw=.7,label='BG subtracted')
    b.set(xlabel=time_label,ylabel='Aperture mean, 0–255 equivalent',title='Intensity without display gain'); b.legend(fontsize=8)
    c.plot(time,r[:,i],lw=.8,c='#8255a0');c.axhline(R_LIMIT,ls=':',c='.5',label='Model r limit')
    c.set(xlabel=time_label,ylabel='r',ylim=(0,max(1.05,float(np.nanmax(r[:,i]))*1.04)) if np.isfinite(r[:,i]).any() else (0,1.05),title='Individual-exposure r')
    for ax in [b,c]: ax.axvline(time[t],c='.5',ls='--',lw=.7)
    fig.suptitle(f"{spot_id} · {s['reason']} · frame {t+1} · {gain}× display · θ={theta_deg[t,i]:.1f}°")
    plt.show()
    print(f"Raw/upper-bound peak={s['raw_peak']:.1f}; saturation flag={s['saturated']}; {saturation_source}")
    print(f"Suppressed by: {s['suppressed_by']}; channels [0,90,45,135]={np.round(channels[t,i],2)}")
    print(f"r={r[t,i]:.4f}; θ={theta_deg[t,i]:.2f}°; φ={phi_deg[t,i]:.2f}°; pair-sum imbalance={imbalance[t,i]:.3f}")
    if pixel_values:
        for title,arr in [(raw_label,raw),('Minimum BG',bg),('After subtraction',corrected)]:
            print(title); print(np.array2string(arr[top:bottom,l:rr],precision=1,threshold=APERTURE**2+1,max_line_width=140))

if spots:
    ordered = sorted(spots,key=lambda s:(s['reason']!='kept',-s['dog_peak']))
    try:
        import ipywidgets as widgets
        picker=widgets.Dropdown(options=[(f"{s['id']} · {s['reason']} · r={s['r']:.3f}",s['id']) for s in ordered],description='Spot')
        frame_slider=widgets.IntSlider(value=FRAME_NUMBER,min=1,max=n_frames,description='Frame',continuous_update=False)
        gain_picker=widgets.Dropdown(options=[1,5,20,50],value=20,description='Display ×')
        controls=widgets.HBox([picker,frame_slider,gain_picker])
        inspection=widgets.interactive_output(inspect_spot,dict(spot_id=picker,frame_number=frame_slider,gain=gain_picker))
        display(controls,inspection)
    except ImportError:
        inspect_spot(ordered[0]['id'],FRAME_NUMBER,DISPLAY_GAIN)
else:
    print("No candidates at these settings. Lower K_SD or check the background and input frame.")

## 5. Population statistics — one value per retained spot

Both panels use the **same chosen exposure**, with equal weight per retained spot. Percentages are counts divided by **all retained detections**, so the θ histogram may total less than 100% when radii are outside the model. The r plot keeps those values.

The dashed line is the exact bin probability for ideal 3-D isotropy, p(θ) = sin θ. It is **not flat** on these axes, and does not correct orientation-dependent detection efficiency. These are ordinary percentages, not solid-angle-normalized densities. No peak fitting or population identification is attempted.

In [ ]:
def population_plot():
    values=r[FRAME_NUMBER-1,keep]
    angles=theta_deg[FRAME_NUMBER-1,keep]
    n=len(values); finite=np.isfinite(values); valid=np.isfinite(angles)
    print(f"Retained: {n}; valid θ: {valid.sum()}; above r limit: {np.sum(values > R_LIMIT)}; undefined X/Y: {np.sum(~finite)}")
    print("Each candidate's saturation flag covers all",n_frames,"exposures; every flagged candidate is excluded.")
    redges=np.linspace(0,1.44,73); tedges=np.arange(0,92,2)
    theta_at_edges=theta_from_radius(np.minimum(redges,R_LIMIT),A,B,C)
    iso_r=np.diff(1-np.cos(theta_at_edges))*100
    iso_t=np.diff(1-np.cos(np.deg2rad(tedges)))*100
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
    for ax,v,edges,iso in [(axes[0],values[finite],redges,iso_r),(axes[1],angles[valid],tedges,iso_t)]:
        counts=np.histogram(v,edges)[0]
        ax.stairs(counts/max(n,1)*100,edges,color='#166888',lw=1.8,label=f'Detections (n={n})')
        ax.stairs(iso,edges,ls='--',color='.5',lw=1,label='Ideal 3-D isotropic')
        ax.yaxis.set_major_formatter(PercentFormatter(100));ax.legend(fontsize=9)
    axes[0].axvline(R_LIMIT,ls=':',color='.4');axes[0].axvspan(R_LIMIT,1.44,color='.5',alpha=.07)
    axes[0].set(xlim=(0,1.44),xlabel='Anisotropy radius r',ylabel='Retained detections per 0.02 bin (%)',title=f'r limit = {R_LIMIT:.3f}')
    axes[1].set(xlim=(0,90),xlabel='θ from surface normal (°)',ylabel='Retained detections per 2° bin (%)')
    fig.suptitle(f'Frame {FRAME_NUMBER} · minimum BG · {K_SD:g} SD · {APERTURE}×{APERTURE} aperture')
    return fig

population_figure=population_plot()
plt.show()

## Optional export

Set `EXPORT = True` to save candidate coordinates, exclusions, selected-frame angles, all-frame channel traces and the histogram/settings. A new numbered folder is created on each export; input data are never changed. For reruns with changed settings, use **Restart Kernel and Run All**.

In [ ]:
EXPORT = False
if EXPORT:
    base=ROOT/'analysis/quickstart_output'/FRAMESTACK.stem
    destination=base; suffix=1
    while destination.exists():
        destination=base.with_name(base.name+f'_{suffix}');suffix+=1
    destination.mkdir(parents=True)
    fields=['id','x','y','area','bounds','bbox','dog_peak','reason','suppressed_by','saturated','raw_peak','r','theta_deg','phi_deg','intensity_8bit_equiv','pair_sum_imbalance']
    with (destination/'spots.csv').open('w',newline='',encoding='utf8') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');writer.writeheader();writer.writerows(spots)
    np.savez_compressed(destination/'traces.npz',channels=channels,raw_or_proxy_channels=raw_channels,xy=xy,r=r,theta_deg=theta_deg,phi_deg=phi_deg,time=time,kept=keep,ids=np.array([s['id'] for s in spots]))
    settings=dict(stack=str(FRAMESTACK),stack_bytes=FRAMESTACK.stat().st_size,stack_mtime_ns=FRAMESTACK.stat().st_mtime_ns,
                  background=str(MINIMUM_BG),background_sha256=bg_file_hash,saved_with_bg=subtracted,measurement_frames=n_frames,
                  frame_number=FRAME_NUMBER,k_sd=K_SD,min_area=MIN_AREA,max_area=MAX_AREA,aperture=APERTURE,
                  origin_xy=origin,exclude_ellipses=EXCLUDE_ELLIPSES,threshold=threshold,model=MODEL,
                  saturation_at=SATURATION_AT,raw_full_scale=RAW_FULL_SCALE,saturation_source=saturation_source,
                  fps=fps,metadata=metadata,notebook_sha256=sha256(ROOT/'Framestack_quickstart.ipynb'),
                  optics_sha256=sha256(ROOT/'src/polarcam/optics.py'),channels_sha256=sha256(ROOT/'src/polarcam/live/intensity_spots.py'),
                  detector_sha256=sha256(ROOT/'src/polarcam/live/Detection_alg_offline.py'))
    (destination/'settings.json').write_text(json.dumps(settings,indent=2),encoding='utf8')
    population_figure.savefig(destination/'theta_statistics.png',dpi=180)
    print('Saved:',destination)
else:
    print('Export off; inputs unchanged.')